# Phase 1 - Data pipeline and business  understanding 

Goal: create a clean analytical foundation and churn analysis using KKbox dataset
- Main objectives: Load and validate datasets
- Understand table relationships
- Audit data quality
- Explore churn behavior
- Build initial business observation

In [ ]:
import pandas as pd
import numpy as np
import gc
import warnings
from pathlib import PathS
warnings.filterwarnings("ignore")
#import of libraries


In [6]:
Raw = Path("../data/raw")
MEMBERS_PATH = Raw / "members_v3.csv"
TRAIN1_PATH = Raw / "train.csv"
TRAIN2_PATH = Raw / "train_v2.csv"
TRANSACTIONS_PATH1 = Raw / "transactions.csv"
TRANSACTIONS_PATH2 = Raw / "transactions_v2.csv"
USER_LOGS_PATH = Raw / "user_logs.csv"
USER_LOGS_V2_PATH = Raw / "user_logs_v2.csv"
TRAIN_PATHS = [TRAIN1_PATH, TRAIN2_PATH]
TRANSACTIONS_PATHS = [TRANSACTIONS_PATH1, TRANSACTIONS_PATH2]

REFERENCE_DATE   = pd.Timestamp("2017-04-01")
CHUNK_SIZE       = 2_000_000
SCALE_POS_WEIGHT = 10
EXPECTED_USERS   = 1_082_190
EXPECTED_CHURN   = 0.0915

PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

train1 = pd.read_csv(TRAIN1_PATH)
train2 = pd.read_csv(TRAIN2_PATH)
train = pd.concat([train1, train2], ignore_index=True).drop_duplicates('msno').reset_index(drop=True)
print(f"Train after dedup: {train.shape[0]:,} users, churn rate: {train['is_churn'].mean():.2%}")

transactions_old = pd.read_csv(TRANSACTIONS_PATH1)
transactions_new = pd.read_csv(TRANSACTIONS_PATH2)
transactions = pd.concat([transactions_old, transactions_new], ignore_index=True)
del transactions_old, transactions_new
gc.collect()
print(f"Transactions loaded: {transactions.shape} — staging copies freed")

Train after dedup: 1,082,190 users, churn rate: 9.15%
Transactions loaded: (22978755, 9) — staging copies freed


In [17]:
all_paths = [MEMBERS_PATH, TRAIN1_PATH, TRAIN2_PATH,
             TRANSACTIONS_PATH1, TRANSACTIONS_PATH2,
             USER_LOGS_PATH, USER_LOGS_V2_PATH]
for p in all_paths:
    status = "✓" if p.exists() else "✗ MISSING"
    print(f"{status}  {p.name}")

✓  members_v3.csv
✓  train.csv
✓  train_v2.csv
✓  transactions.csv
✓  transactions_v2.csv
✓  user_logs.csv
✓  user_logs_v2.csv


In [18]:
members = pd.read_csv(MEMBERS_PATH)

members['registration_init_time'] = pd.to_datetime(
    members['registration_init_time'].astype(str),
    format='%Y%m%d',
    errors='coerce'
)

birth_valid = members['bd'].between(1940, 2006)
members.loc[~birth_valid, 'bd'] = np.nan

print(f"Members loaded: {members.shape}")
print(f"registration_init_time dtype: {members['registration_init_time'].dtype}")
print(f"bd: {birth_valid.sum():,} valid, {(~birth_valid).sum():,} set to NaN")



Members loaded: (6769473, 6)
registration_init_time dtype: datetime64[ns]
bd: 6 valid, 6,769,467 set to NaN


In [9]:
reg_dates = members.set_index('msno')['registration_init_time'].to_dict()
reg_dates_s = pd.Series(reg_dates, dtype='datetime64[ns]')
print(f"Registration lookup built: {len(reg_dates_s):,} users")

Registration lookup built: 6,769,473 users


In [ ]:
customer_txn_count = (
    transactions.groupby("msno")
    .size()
    .reset_index(name="txn_count")
)

In [10]:
chunk_accum = []
total_chunks = 0

for filepath in [USER_LOGS_PATH, USER_LOGS_V2_PATH]:
    print(f"\nProcessing: {filepath.name}")
    for chunk in pd.read_csv(filepath, chunksize=CHUNK_SIZE):
        total_chunks += 1

        # Date parsing
        chunk['date'] = pd.to_datetime(
            chunk['date'].astype(str), format='%Y%m%d', errors='coerce'
        )
        chunk = chunk.dropna(subset=['date'])

        # Cap impossible values
        chunk['total_secs'] = chunk['total_secs'].clip(upper=86_400)
        chunk['num_unq']    = chunk['num_unq'].clip(lower=0, upper=10_000)

        # Completion rate
        plays = chunk[['num_25','num_50','num_75','num_985','num_100']].sum(axis=1)
        plays = plays.replace(0, np.nan)
        chunk['completion_rate'] = (chunk['num_100'] / plays).fillna(0.0)

        # First-week tag
        chunk['reg_date']       = chunk['msno'].map(reg_dates_s)
        chunk['days_since_reg'] = (chunk['date'] - chunk['reg_date']).dt.days
        is_first_week           = chunk['days_since_reg'].between(0, 6)

        # Aggregate all rows
        agg_all = chunk.groupby('msno', sort=False).agg(
            date_max       = ('date',            'max'),
            date_count     = ('date',            'count'),
            num_unq_sum    = ('num_unq',         'sum'),
            total_secs_sum = ('total_secs',      'sum'),
            completion_sum = ('completion_rate', 'sum'),
        ).reset_index()

        # Aggregate first-week rows
        fw = chunk[is_first_week].groupby('msno', sort=False).agg(
            fw_num_unq_sum = ('num_unq', 'sum'),
            fw_day_count   = ('date',    'count'),
        ).reset_index()

        agg_all = agg_all.merge(fw, on='msno', how='left')
        agg_all['fw_num_unq_sum'] = agg_all['fw_num_unq_sum'].fillna(0)
        agg_all['fw_day_count']   = agg_all['fw_day_count'].fillna(0)

        chunk_accum.append(agg_all)

        if total_chunks % 5 == 0:
            print(f"  Chunk {total_chunks} done")

print(f"\nTotal chunks processed: {total_chunks}")


Processing: user_logs.csv
  Chunk 5 done
  Chunk 10 done
  Chunk 15 done
  Chunk 20 done
  Chunk 25 done
  Chunk 30 done
  Chunk 35 done
  Chunk 40 done
  Chunk 45 done
  Chunk 50 done
  Chunk 55 done
  Chunk 60 done
  Chunk 65 done
  Chunk 70 done
  Chunk 75 done
  Chunk 80 done
  Chunk 85 done
  Chunk 90 done
  Chunk 95 done
  Chunk 100 done
  Chunk 105 done
  Chunk 110 done
  Chunk 115 done
  Chunk 120 done
  Chunk 125 done
  Chunk 130 done
  Chunk 135 done
  Chunk 140 done
  Chunk 145 done
  Chunk 150 done
  Chunk 155 done
  Chunk 160 done
  Chunk 165 done
  Chunk 170 done
  Chunk 175 done
  Chunk 180 done
  Chunk 185 done
  Chunk 190 done
  Chunk 195 done

Processing: user_logs_v2.csv
  Chunk 200 done
  Chunk 205 done

Total chunks processed: 207


In [11]:
print("combining chunks")
ul_raw = pd.concat(chunk_accum, ignore_index=True)
del chunk_accum
gc.collect()
#collapse to one row per user
ul_agg = ul_raw.groupby('msno', sort=False).agg(
    last_active_date = ('date_max', 'max'),
    activity_day_count = ('date_count', 'sum'),
    num_unq_sum = ('num_unq_sum', 'sum'),
    total_secs_sum = ('total_secs_sum', 'sum'),
    completion_sum = ('completion_sum', 'sum'),
    fw_num_unq_sum = ('fw_num_unq_sum', 'sum'),
    fw_day_count = ('fw_day_count', 'sum'),
).reset_index()
del ul_raw
gc.collect()
#compute metrics
ul_agg['avg_num_unq']         = (ul_agg['num_unq_sum']   / ul_agg['activity_day_count']).round(2)
ul_agg['avg_total_secs']      = (ul_agg['total_secs_sum'] / ul_agg['activity_day_count']).round(1)
ul_agg['avg_completion_rate'] = (ul_agg['completion_sum'] / ul_agg['activity_day_count']).round(4)
ul_agg['first_week_depth'] = (
    ul_agg['fw_num_unq_sum'] / ul_agg['fw_day_count'].replace(0, np.nan)
).fillna(0).round(2)
assert ul_agg['msno'].nunique() == ul_agg.shape[0],"Duplicate msno in ul_agg"
assert ul_agg['avg_total_secs'].max() <= 86_400, "clipping err"
print(f"ul_agg ready: {ul_agg.shape}")
print(ul_agg[['avg_num_unq', 'avg_total_secs', 'avg_completion_rate', 'first_week_depth']].describe())


combining chunks
ul_agg ready: (5339422, 12)
        avg_num_unq  avg_total_secs  avg_completion_rate  first_week_depth
count  5.339422e+06    5.339422e+06         5.339422e+06      5.339422e+06
mean   1.676728e+01   -3.198590e+11         4.800362e-01      1.136571e+01
std    1.821465e+01    1.392837e+13         3.160180e-01      1.866114e+01
min    1.000000e+00   -9.223372e+15         0.000000e+00      0.000000e+00
25%    4.090909e+00    6.002671e+02         1.989319e-01      0.000000e+00
50%    1.181768e+01    2.308718e+03         5.274106e-01      4.000000e+00
75%    2.300000e+01    5.214801e+03         7.419765e-01      1.514000e+01
max    1.402725e+03    8.640000e+04         1.000000e+00      8.530000e+02


In [15]:
#
#
master = train.copy()
before = master.shape[0]
master = master.merge(members, on='msno', how='left')
assert master.shape[0] == before, "Members join changed row count"
master = master.merge(transactions_features, on='msno', how='left')
assert master.shape[0] == before, "Transactions join changed row count"
master = master.merge(
    ul_agg[['msno', 'last_active_date', 'activity_day_count',
            'avg_num_unq', 'avg_total_secs', 'avg_completion_rate', 'first_week_depth']],
    on='msno', how='left'
)
assert master.shape[0] == before, "User logs join changed row count"
#flag missing
master['has_tx'] = master['total_paid'].notna().astype(int)
master['has_logs'] = master['avg_num_unq'].notna().astype(int)
#derived time features
master['tenure_days']           = (REFERENCE_DATE - master['registration_init_time']).dt.days
master['days_since_last_active'] = (REFERENCE_DATE - master['last_active_date']).dt.days
master['revenue_per_day'] = (
    master['total_paid'] / master['tenure_days'].replace(0, np.nan)
).round(4)
print(f"Master dataset ready: {master.shape}")
print(f"has_tx=1: {master['has_tx'].sum():,} users have transaction data")
print(f"has_logs=1: {master['has_logs'].sum():,} users have log data")



KeyError: 'total_paid'

In [26]:
print("=" * 50)
print("Data Pipeline Initialized")
print("=" * 50)
#Row count check
assert master.shape[0] == EXPECTED_USERS, \
    f"FAIL: {master.shape[0]:,} rows is not equal to expected {EXPECTED_USERS:,}"
print(f"check Row count: {master.shape[0]:,}")

#every id must be unique
assert master['msno'].nunique() == master.shape[0], \
    f"FAIL: Duplicate msno found"
print(f"correct, every msno is unique: {master['msno'].nunique():,}")
#churn rate check (must be within 0.2% of expected)
assert abs(master['is_churn'].mean() - EXPECTED_CHURN) < 0.002, \
    f"FAIL: Churn rate {master['is_churn'].mean():.4%} is not within 0.2% of expected {EXPECTED_CHURN:.4%}"
print(f"correct, churn rate is within 0.2% of expected: {master['is_churn'].mean():.2%}")

#Check of negative tenure_days
assert (master['tenure_days'].fillna(0) >= 0).all(), \
    f"FAIL: Negative tenure_days found"
print(f"correct, no negative tenure_days found")

#null rates
null_rates = master.isnull().mean().sort_values(ascending=False)
for col, rate in null_rates[null_rates > 0].items():
    print(f"WARNING: {col} has {rate:.1%} ")

print("Data pipeline completed successfully.")

Data Pipeline Initialized
check Row count: 1,082,190
correct, every msno is unique: 1,082,190
correct, churn rate is within 0.2% of expected: 9.15%


KeyError: 'tenure_days'

In [ ]:
MASTER_PATH = PROCESSED / "master.csv"
master.to_csv(MASTER_PATH, index=False)
verify = pd.read_csv(MASTER_PATH, usecols=[ 'msno', 'is_churn' ])
assert verify.shape[0] == EXPECTED_USERS, "save verification failed"
print(f"master.csv saved")
print(f"Rows: {master.shape[0]:,}, Columns: {master.shape[1]:,}")
print(f" Size: {MASTER_PATH.stat().st_size / 1e6:.1f} MB  ")
print(f"{MASTER_PATH}")

In [ ]:
members.head()

,msno,city,bd,gender,registered_via,registration_init_time
0,Rb9UwLQTrxzBVwCB6+bCcSQWZ9JiNLC9dXtM1oEsZA8=,1,0,NaN,11,20110911
1,+tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=,1,0,NaN,7,20110914
2,cV358ssn7a0f7jZOwGNWS07wCKVqxyiImJUX6xcIwKw=,1,0,NaN,11,20110915
3,9bzDeJP6sQodK73K5CBlJ6fgIQzPeLnRl0p5B77XP+g=,1,0,NaN,11,20110915
4,WFLY3s7z4EZsieHCt63XrsdtfTEmJ+2PnnKLH5GY4Tk=,6,32,female,9,20110915


In [ ]:
members.shape

(6769473, 6)

In [ ]:
members.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6769473 entries, 0 to 6769472
Data columns (total 6 columns):
 #   Column                  Dtype 
---  ------                  ----- 
 0   msno                    object
 1   city                    int64 
 2   bd                      int64 
 3   gender                  object
 4   registered_via          int64 
 5   registration_init_time  int64 
dtypes: int64(4), object(2)
memory usage: 309.9+ MB


In [ ]:
members.isnull().sum()

msno                            0
city                            0
bd                              0
gender                    4429505
registered_via                  0
registration_init_time          0
dtype: int64

## Initial observation: members dataset
- Dataset contains large amount of rows
- Gender of 65.4% of members is unknown


In [ ]:

print(f"Train data loaded: {train.shape}")

Train data loaded: (1082190, 2)


In [ ]:
train.head()


,msno,is_churn
0,waLDQMmcOu2jLDaV1ddDkgCrB/jl6sD66Xzs0Vqax1Y=,1
1,QA7uiXy8vIbUSPOkCf9RwQ3FsT8jVq2OxDr8zqa7bRQ=,1
2,fGwBva6hikQmTJzrbz/2Ezjm5Cth5jZUNvXigKK2AFA=,1
3,mT5V8rEpa+8wuqi6x0DoVd3H5icMKkE9Prt49UlmK+4=,1
4,XaPhtGLk/5UvvOYHcONTwsnH97P4eGECeq+BARGItRw=,1


In [ ]:
train["is_churn"].value_counts()

is_churn
0    983162
1     99028
Name: count, dtype: int64

In [ ]:
churn_rate = train["is_churn"].mean()
print(f"Churn Rate: {churn_rate:.2%}")

Churn Rate: 9.15%


In [ ]:
overlap = set(train1['msno']) & set(train2['msno'])
print(f"Users in both train files: {len(overlap):,}")
combined_dedup = pd.concat([train1, train2]).drop_duplicates('msno')
print(f"After dedup: {combined_dedup.shape[0]:,} users")
print(f"Churn rate after dedup: {combined_dedup['is_churn'].mean():.4f}")

Users in both train files: 881,701
After dedup: 1,082,190 users
Churn rate after dedup: 0.0915


## Observation - Churn labels
- Churn rate = 9,15%, relatively small, only small fraction churned
- dataset is imbalanced
- prediction models may bias

In [ ]:
train["is_churn"].value_counts()
train["is_churn"].value_counts(normalize=True)

is_churn
0    0.908493
1    0.091507
Name: proportion, dtype: float64

In [ ]:

try:
    print(f"Combined transactions data loaded: {transactions.shape}")
    print(f"From {transactions['msno'].nunique():,} unique members")
    print(f"Source counts: {len(transactions_old):,} + {len(transactions_new):,} -> {len(transactions):,}")
except NameError:
    print("transactions not defined yet. Reloading now...")
    transactions_old = pd.read_csv(TRANSACTIONS_PATH1)
    transactions_new = pd.read_csv(TRANSACTIONS_PATH2)
    transactions = pd.concat([transactions_old, transactions_new], ignore_index=True)
    print(f"Reloaded combined transactions data: {transactions.shape}")
    print(f"From {transactions['msno'].nunique():,} unique members")
    print(f"Source counts: {len(transactions_old):,} + {len(transactions_new):,} -> {len(transactions):,}")

Combined transactions data loaded: (22978755, 9)
From 2,426,143 unique members
Source counts: 21,547,746 + 1,431,009 -> 22,978,755


In [ ]:
transactions.head()

,msno,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date,is_cancel
0,YyO+tlZtAXYXoZhNr3Vg3+dfVQvrBVGO8j1mfqe4ZHc=,41,30,129,129,1,2015-09-30,2015-11-01,0
1,AZtu6Wl0gPojrEQYB8Q3vBSmE2wnZ3hi1FbK1rQQ0A4=,41,30,149,149,1,2015-09-30,2015-10-31,0
2,UkDFI97Qb6+s2LWcijVVv4rMAsORbVDT2wNXF0aVbns=,41,30,129,129,1,2015-09-30,2016-04-27,0
3,M1C56ijxozNaGD0t2h68PnH2xtx5iO5iR2MVYQB6nBI=,39,30,149,149,1,2015-09-30,2015-11-28,0
4,yvj6zyBUaqdbUQSrKsrZ+xNDVM62knauSZJzakS9OW4=,39,30,149,149,1,2015-09-30,2015-11-21,0


In [ ]:
transactions.shape

(22978755, 9)

In [ ]:
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22978755 entries, 0 to 22978754
Data columns (total 9 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   msno                    object        
 1   payment_method_id       int64         
 2   payment_plan_days       int64         
 3   plan_list_price         int64         
 4   actual_amount_paid      int64         
 5   is_auto_renew           int64         
 6   transaction_date        datetime64[ns]
 7   membership_expire_date  datetime64[ns]
 8   is_cancel               int64         
dtypes: datetime64[ns](2), int64(6), object(1)
memory usage: 1.5+ GB


In [ ]:
transactions.isnull().sum()

msno                      0
payment_method_id         0
payment_plan_days         0
plan_list_price           0
actual_amount_paid        0
is_auto_renew             0
transaction_date          0
membership_expire_date    0
is_cancel                 0
dtype: int64

In [ ]:
members["msno"].nunique()

6769473

In [ ]:

print(f"Total transactions rows: {transactions.shape[0]}")

Total transactions rows: 22978755


In [ ]:
transactions["msno"].nunique()

2426143

In [ ]:
sample = pd.read_csv(USER_LOGS_PATH, nrows=1000)
print(sample.head())
sample.shape

                                           msno      date  num_25  num_50  \
0  rxIP2f2aN0rYNp+toI0Obt/N/FYQX8hcO1fTmmy2h34=  20150513       0       0   
1  rxIP2f2aN0rYNp+toI0Obt/N/FYQX8hcO1fTmmy2h34=  20150709       9       1   
2  yxiEWwE9VR5utpUecLxVdQ5B7NysUPfrNtGINaM2zA8=  20150105       3       3   
3  yxiEWwE9VR5utpUecLxVdQ5B7NysUPfrNtGINaM2zA8=  20150306       1       0   
4  yxiEWwE9VR5utpUecLxVdQ5B7NysUPfrNtGINaM2zA8=  20150501       3       0   

   num_75  num_985  num_100  num_unq  total_secs  
0       0        0        1        1     280.335  
1       0        0        7       11    1658.948  
2       0        0       68       36   17364.956  
3       1        1       97       27   24667.317  
4       0        0       38       38    9649.029  


(1000, 9)

In [ ]:
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"], format="%Y%m%d", errors="coerce")
transactions["membership_expire_date"] = pd.to_datetime(transactions["membership_expire_date"], format="%Y%m%d", errors="coerce")

In [27]:
LP = (
    transactions.groupby("msno")["transaction_date"].max().reset_index(name="last_transaction_date")
)

In [19]:
LTV = (
    transactions.groupby("msno")["actual_amount_paid"]
    .sum()
    .reset_index(name="total_paid")
)  


In [20]:
avg_duration = (
    transactions.groupby("msno")["payment_plan_days"]
    .mean()
    .reset_index(name="avg_plan_duration")
)  

In [21]:
auto_renew_history= (
    transactions.groupby("msno")["is_auto_renew"]
    .max()
    .reset_index(name="auto_renew_history")
)

In [22]:
cancel_count = (
    transactions.groupby("msno")["is_cancel"]
    .sum()
    .reset_index(name="cancel_count")
)
cancel_count.head()
cancel_count.shape
cancel_count.info()
cancel_count["cancel_count"].describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2426143 entries, 0 to 2426142
Data columns (total 2 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   msno          object
 1   cancel_count  int64 
dtypes: int64(1), object(1)
memory usage: 37.0+ MB


count    2.426143e+06
mean     3.676552e-01
std      5.772337e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      2.100000e+01
Name: cancel_count, dtype: float64

In [23]:
last_expire = (
    transactions.groupby("msno")["membership_expire_date"]
    .max()
    .reset_index(name="last_expire_date")
)

In [25]:
transactions_features = LP
transactions_features = transactions_features.merge(LTV, on="msno", how="left")
transactions_features = transactions_features.merge(avg_duration, on="msno", how="left")
transactions_features = transactions_features.merge(auto_renew_history, on="msno", how="left")
transactions_features = transactions_features.merge(cancel_count, on="msno", how="left")
transactions_features = transactions_features.merge(last_expire, on="msno", how="left")
transactions_features = transactions_features.merge(customer_txn_count, on="msno", how="left")

assert transactions_features['msno'].nunique() == transactions_features.shape[0], "Duplicate msno — merge went wrong"
assert transactions_features.shape[0] == 2_426_143, f"Wrong row count: {transactions_features.shape[0]}"
print(f"✓ transactions_features ready: {transactions_features.shape}")
print(transactions_features.dtypes)



NameError: name 'customer_txn_count' is not defined

In [ ]:
transactions_features.head()
transactions_features.shape
transactions_features.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2426143 entries, 0 to 2426142
Data columns (total 6 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   msno                   object        
 1   last_transaction_date  datetime64[ns]
 2   total_paid             int64         
 3   avg_plan_duration      float64       
 4   auto_renew_rate        int64         
 5   cancel_count           int64         
dtypes: datetime64[ns](1), float64(1), int64(3), object(1)
memory usage: 111.1+ MB
